# Part 2: MIMIC-IV Demo - SQL Exploration
Goal: practice SQL on real EHR-structured tables (admissions, diagnoses, prescriptions).

In [1]:
# Step 10.0: Locate the MIMIC demo folder relative to the project root
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent          # notebooks/ -> project root
MIMIC_DIR = PROJECT_ROOT / "data" / "mimic-iv-clinical-database-demo-2.2"
DB_PATH = PROJECT_ROOT / "mimic_demo.db"
print("MIMIC folder exists:", MIMIC_DIR.exists())

# List every data file with its size (hosp / icu)
for p in sorted(MIMIC_DIR.rglob("*.csv")):
    print(p.relative_to(MIMIC_DIR), "-", round(p.stat().st_size / 1024), "KB")

MIMIC folder exists: True
demo_subject_id.csv - 1 KB
hosp\admissions.csv - 46 KB
hosp\d_hcpcs.csv - 3252 KB
hosp\d_icd_diagnoses.csv - 8642 KB
hosp\d_icd_procedures.csv - 7169 KB
hosp\d_labitems.csv - 62 KB
hosp\diagnoses_icd.csv - 126 KB
hosp\drgcodes.csv - 32 KB
hosp\emar.csv - 5113 KB
hosp\emar_detail.csv - 7068 KB
hosp\hcpcsevents.csv - 4 KB
hosp\labevents.csv - 11978 KB
hosp\microbiologyevents.csv - 639 KB
hosp\omr.csv - 120 KB
hosp\patients.csv - 3 KB
hosp\pharmacy.csv - 3313 KB
hosp\poe.csv - 4337 KB
hosp\poe_detail.csv - 230 KB
hosp\prescriptions.csv - 3054 KB
hosp\procedures_icd.csv - 28 KB
hosp\provider.csv - 277 KB
hosp\services.csv - 14 KB
hosp\transfers.csv - 100 KB
icu\caregiver.csv - 89 KB
icu\chartevents.csv - 63185 KB
icu\d_items.csv - 360 KB
icu\datetimeevents.csv - 1631 KB
icu\icustays.csv - 22 KB
icu\ingredientevents.csv - 4380 KB
icu\inputevents.csv - 5218 KB
icu\outputevents.csv - 789 KB
icu\procedureevents.csv - 267 KB


In [2]:
# Step 10.1: Build the SQLite database only if it does not exist yet
import sqlite3
import pandas as pd
import numpy as np
REBUILD = False          # set True to rebuild the database from the CSV files
if REBUILD and DB_PATH.exists():
    DB_PATH.unlink()
if DB_PATH.exists():
    print("Database already exists - skipping load.")
else:
    conn = sqlite3.connect(DB_PATH)
    for p in sorted(MIMIC_DIR.rglob("*.csv")):
        if p.name == "demo_subject_id.csv":
            continue
        df_t = pd.read_csv(p, low_memory=False)
        df_t.to_sql(p.stem, conn, if_exists="replace", index=False)
        print(f"{p.stem:25s} {len(df_t):>8,} rows")
    conn.close()

Database already exists - skipping load.


In [3]:
# Step 14.1: Validate the database against the source CSV files.
# Set REBUILD = True (above, in Step 10.1) after replacing the CSVs, otherwise a stale .db would be reused.
# Row counts in SQLite must equal the number of data lines in each CSV.
rows = []
for p in sorted(MIMIC_DIR.rglob("*.csv")):
    if p.name == "demo_subject_id.csv":
        continue
    with open(p, "rb") as f:
        n_csv = sum(1 for _ in f) - 1          # minus header (note: assumes no multi-line fields)
    n_db = q(f"SELECT COUNT(*) AS n FROM {p.stem}")["n"][0]
    rows.append({"table": p.stem, "csv_rows": n_csv, "db_rows": n_db, "match": n_csv == n_db})
check = pd.DataFrame(rows)
display(check[~check["match"]])
print("Tables checked:", len(check), "| mismatches:", (~check["match"]).sum())

NameError: name 'q' is not defined

In [ ]:
# Step 14.2: Validate the 30-day readmission logic.
# Each hadm_id is one hospitalization, so transfers inside a stay do not create new admissions here.
# We check for data problems, then compute the rate again excluding planned readmissions.
print("Integrity checks:")
display(q("""
WITH o AS (
  SELECT subject_id, hadm_id, admittime, dischtime, admission_type,
         LEAD(admittime) OVER (PARTITION BY subject_id ORDER BY admittime) AS next_admit
  FROM admissions)
SELECT COUNT(*) AS admissions,
       SUM(dischtime IS NULL) AS missing_discharge,
       SUM(julianday(dischtime) < julianday(admittime)) AS negative_stay,
       SUM(julianday(next_admit) < julianday(dischtime)) AS overlapping_next_admit
FROM o
"""))

print("Readmission: all vs excluding planned next admissions (ELECTIVE / SURGICAL SAME DAY):")
display(q("""
WITH o AS (
  SELECT subject_id, hospital_expire_flag, dischtime,
         LEAD(admittime)       OVER (PARTITION BY subject_id ORDER BY admittime) AS next_admit,
         LEAD(admission_type)  OVER (PARTITION BY subject_id ORDER BY admittime) AS next_type
  FROM admissions)
SELECT COUNT(*) AS discharged_alive,
       SUM(julianday(next_admit) - julianday(dischtime) BETWEEN 0 AND 30) AS readmit_any,
       SUM(julianday(next_admit) - julianday(dischtime) BETWEEN 0 AND 30
           AND next_type NOT IN ('ELECTIVE', 'SURGICAL SAME DAY ADMISSION')) AS readmit_unplanned
FROM o
WHERE hospital_expire_flag = 0
"""))

In [ ]:
# Step 14.3: Orders (prescriptions) versus administrations (emar) for insulin.
# An order does not prove the patient received the drug. emar records the administration events.
print("emar event types:")
display(q("SELECT event_txt, COUNT(*) AS n FROM emar GROUP BY event_txt ORDER BY n DESC LIMIT 10"))

display(q("""
WITH ordered AS (
  SELECT DISTINCT hadm_id FROM prescriptions WHERE LOWER(drug) LIKE '%insulin%'),
administered AS (
  SELECT DISTINCT hadm_id FROM emar
  WHERE LOWER(medication) LIKE '%insulin%' AND event_txt = 'Administered'),
emar_adm AS (SELECT DISTINCT hadm_id FROM emar)
SELECT
  (SELECT COUNT(*) FROM ordered)                                          AS adm_with_insulin_order,
  (SELECT COUNT(*) FROM administered)                                     AS adm_with_insulin_administered,
  (SELECT COUNT(*) FROM ordered o JOIN administered a USING (hadm_id))    AS both,
  (SELECT COUNT(*) FROM ordered o WHERE o.hadm_id IN (SELECT hadm_id FROM emar_adm)) AS ordered_and_in_emar
"""))

### Observation: Database and readmission validation
- All 31 tables match their source CSV files row for row.
- Integrity checks on 275 admissions: no missing discharge times, no negative stays, no overlapping next admissions.
- 30-day readmission after live discharge: 53 of 260 (20.4%) for any next admission and 49 of 260 (18.8%) excluding
  planned next admissions (elective or same-day surgical); approximate 95% interval for the latter 15% to 24%.
  Small demo sample, so illustrative only.

### Observation: Insulin orders versus administrations
- 134 admissions have an insulin order (prescriptions) but only 83 have a recorded insulin administration (emar);
  72 have both. Only 86 of the 134 ordered admissions appear in emar at all, and 84% of those (72 of 86) have an
  administration record.
- emar does not cover every care area (ICU medications are largely recorded elsewhere), so the gap does not mean patients
  were not treated. It shows that orders overstate exposure and that emar coverage is partial.
- 11 admissions have an insulin administration without a name-matched order (possible brand-name or table differences).
- Not Given events are common (2,486 plus 908 sliding-scale), another reason an order is not proof of administration.

## Step 10.2: SQL exploration
SQLite database built from the MIMIC-IV demo (100 patients, 275 admissions).
Because the sample is tiny, results are exploratory, not generalizable.

In [ ]:
# Step 10.2: Reconnect to the database and create a small helper that runs SQL
# and returns the result as a pandas table.
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)

def q(sql):
    return pd.read_sql_query(sql, conn)

# List all tables (SQLite keeps this list in a special table)
q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").T

In [ ]:
# Step 10.3: Always check the column names before writing queries
for t in ["patients", "admissions", "diagnoses_icd", "d_icd_diagnoses", "prescriptions"]:
    cols = q(f"PRAGMA table_info({t})")["name"].tolist()
    print(f"\n{t}:\n  {cols}")

In [ ]:
# Query 1: Most frequent diagnoses.
# diagnoses_icd stores only a code (icd_code). d_icd_diagnoses translates it to a name.
# We JOIN on BOTH icd_code and icd_version because the same code can exist in ICD-9 and ICD-10.
# COUNT(DISTINCT hadm_id) = number of hospital admissions with that diagnosis.
q("""
SELECT d.icd_code,
       d.icd_version,
       dd.long_title            AS diagnosis,
       COUNT(DISTINCT d.hadm_id) AS admissions
FROM diagnoses_icd d
JOIN d_icd_diagnoses dd
  ON d.icd_code = dd.icd_code AND d.icd_version = dd.icd_version
GROUP BY d.icd_code, d.icd_version, dd.long_title
ORDER BY admissions DESC
LIMIT 10
""")

In [ ]:
# Query 2: Most prescribed drugs.
# prescriptions has one row per drug order. We count total orders and distinct admissions.
q("""
SELECT drug,
       COUNT(*)                  AS orders,
       COUNT(DISTINCT hadm_id)   AS admissions
FROM prescriptions
WHERE drug IS NOT NULL
GROUP BY drug
ORDER BY orders DESC
LIMIT 15
""")

In [ ]:
# Query 3: Length of stay by admission type.
# SQLite can subtract dates with julianday(): result is in days.
# The WHERE removes rows with missing discharge time.
q("""
SELECT admission_type,
       COUNT(*)                                                       AS admissions,
       ROUND(AVG(julianday(dischtime) - julianday(admittime)), 1)    AS avg_los_days,
       ROUND(AVG(hospital_expire_flag) * 100, 1)                      AS in_hospital_death_pct
FROM admissions
WHERE dischtime IS NOT NULL
GROUP BY admission_type
ORDER BY admissions DESC
""")

### Observation: Most frequent diagnoses
- Chronic conditions dominate: hypertension (68 of 275 admissions), hyperlipidemia, hypothyroidism, atrial fibrillation.
- The same disease appears under ICD-9 and ICD-10 codes (hyperlipidemia: E785 and 2724),
  so counts must be grouped by condition, not by raw code.
- Diabetes appears as 25000 and as "long-term insulin use" (Z794), relevant to the diabetes project.

### Observation: Most prescribed drugs
- Insulin is the most ordered drug (915 orders, 130 admissions).
- IV fluids, flushes and the generic label "Bag" fill the list; they are not therapeutic drugs
  and are filtered in the next query. This is a data quality issue in free-text drug names.

### Observation: Length of stay
- Average stay ranges from about 1 day (observation) to about 10 days (urgent).
- Groups with fewer than 40 admissions (e.g. urgent: n=38, 13.2% in-hospital deaths) are too small to compare.

In [ ]:
# Query 4: Therapeutic drugs only. We exclude IV fluids, flushes and the generic "Bag".
q("""
SELECT drug,
       COUNT(*)                AS orders,
       COUNT(DISTINCT hadm_id) AS admissions
FROM prescriptions
WHERE drug IS NOT NULL
  AND drug NOT LIKE '%Sodium Chloride%'
  AND drug NOT LIKE '%Dextrose%'
  AND drug NOT LIKE '%Flush%'
  AND drug NOT LIKE '%Water%'
  AND drug NOT IN ('Bag', 'Iso-Osmotic Dextrose')
GROUP BY drug
ORDER BY admissions DESC
LIMIT 15
""")

In [ ]:
# Query 5 (corrected): drugs in diabetic vs non-diabetic admissions.
# ICD-10 diabetes = E08, E09, E10, E11, E13 (NOT E15/E16, which are non-diabetic hypoglycemia).
# ICD-9 diabetes = 250.xx
diab_sql = """
    SELECT DISTINCT hadm_id FROM diagnoses_icd
    WHERE (icd_version = 9 AND icd_code LIKE '250%')
       OR (icd_version = 10 AND (icd_code LIKE 'E08%' OR icd_code LIKE 'E09%'
                              OR icd_code LIKE 'E10%' OR icd_code LIKE 'E11%'
                              OR icd_code LIKE 'E13%'))
"""
n_all = q("SELECT COUNT(DISTINCT hadm_id) AS n FROM admissions")["n"][0]
n_dm  = q(f"SELECT COUNT(*) AS n FROM ({diab_sql})")["n"][0]
print("Admissions:", n_all, "| with diabetes:", n_dm)

drugs = q(f"""
WITH diabetic AS ({diab_sql})
SELECT p.drug,
       COUNT(DISTINCT p.hadm_id) AS all_adm,
       COUNT(DISTINCT CASE WHEN d.hadm_id IS NOT NULL THEN p.hadm_id END) AS dm_adm
FROM prescriptions p
LEFT JOIN diabetic d ON p.hadm_id = d.hadm_id
WHERE p.drug IS NOT NULL
  AND p.drug NOT LIKE '%Sodium Chloride%' AND p.drug NOT LIKE '%Dextrose%'
  AND p.drug NOT LIKE '%Flush%' AND p.drug <> 'Bag'
GROUP BY p.drug
HAVING all_adm >= 40
""")

# Share of diabetic admissions that got the drug vs share of non-diabetic admissions
drugs["pct_diabetic"]     = (drugs["dm_adm"] / n_dm * 100).round(0)
drugs["pct_non_diabetic"] = ((drugs["all_adm"] - drugs["dm_adm"]) / (n_all - n_dm) * 100).round(0)
drugs["gap"] = drugs["pct_diabetic"] - drugs["pct_non_diabetic"]
drugs.sort_values("gap", ascending=False).head(12)

### Observation: Drugs in diabetic vs non-diabetic admissions
- 112 of 275 admissions (41%) carry a diabetes diagnosis (ICD-9 250.xx; ICD-10 E08-E11, E13).
- Glucose-management drugs separate the groups most: glucagon (84% vs 22%), insulin (82% vs 23%),
  glucose gel (72% vs 13%).
- Cardioprotective drugs are also more common in diabetic admissions: atorvastatin (41% vs 12%), aspirin (51% vs 26%).
- About 23% of admissions without a diabetes code received insulin. Possible reasons: stress or
  steroid-induced hyperglycemia, or diabetes that is present but not coded. Not testable with this data.
- Limitations: orders are not administrations (`emar` is needed for that), the sample is small
  (275 admissions), and the comparison is descriptive with no significance testing.

In [ ]:
# Query 6: 30-day readmission rate in MIMIC, using a window function.
# LEAD() looks at the NEXT admission of the same patient (ordered by admission time).
# We keep only admissions where the patient left alive, same logic as in the diabetes project.
q("""
WITH ordered AS (
    SELECT subject_id, hadm_id, admittime, dischtime, hospital_expire_flag,
           LEAD(admittime) OVER (PARTITION BY subject_id ORDER BY admittime) AS next_admit
    FROM admissions
)
SELECT COUNT(*) AS discharged_alive,
       SUM(CASE WHEN julianday(next_admit) - julianday(dischtime) BETWEEN 0 AND 30
                THEN 1 ELSE 0 END) AS readmit_30,
       ROUND(100.0 * SUM(CASE WHEN julianday(next_admit) - julianday(dischtime) BETWEEN 0 AND 30
                              THEN 1 ELSE 0 END) / COUNT(*), 1) AS readmit_rate_pct
FROM ordered
WHERE hospital_expire_flag = 0
""")

### Observation: Most prescribed therapeutic drugs
- After removing IV fluids and flushes, the most common orders are routine supportive care:
  acetaminophen (176 of 275 admissions), heparin (173), docusate (155), senna (148).
- Insulin appears in 130 admissions (47%) with about 7 orders per admission (915 orders),
  consistent with repeated dose orders and titration. Glucagon has the same admission count but only
  about 1.1 orders per admission, consistent with a single standing hypoglycemia-protocol order.
- Caution: `prescriptions` contains orders, not administrations. Actual administration needs `emar`.

### Observation: 30-day readmission in the MIMIC demo
- 53 of 260 live discharges were followed by another admission within 30 days: 20.4% (95% CI about 16% to 26%).
- This is much higher than the 9.0% in the diabetes dataset, but the two are not comparable:
  different hospital and population (tertiary academic center), tiny sample, all admission types counted,
  and every admission per patient counted (the diabetes project kept the first encounter only).

In [ ]:
# Step 19.1: Export small AGGREGATE tables for the dashboard (counts only, no patient-level rows)
import json
APP_DATA = PROJECT_ROOT / "app_data"
APP_DATA.mkdir(exist_ok=True)

# 1) Most frequent diagnoses. ICD-9 and ICD-10 versions of one disease appear separately.
q("""
SELECT d.icd_version, d.icd_code, dd.long_title AS diagnosis,
       COUNT(DISTINCT d.hadm_id) AS admissions
FROM diagnoses_icd d
JOIN d_icd_diagnoses dd ON d.icd_code = dd.icd_code AND d.icd_version = dd.icd_version
GROUP BY d.icd_version, d.icd_code, dd.long_title
ORDER BY admissions DESC LIMIT 15
""").to_csv(APP_DATA / "mimic_diagnoses.csv", index=False)

# 2) Most ordered therapeutic drugs (name-based filter for fluids and flushes: a heuristic)
q("""
SELECT drug, COUNT(*) AS orders, COUNT(DISTINCT hadm_id) AS admissions
FROM prescriptions
WHERE drug IS NOT NULL
  AND drug NOT LIKE '%Sodium Chloride%' AND drug NOT LIKE '%Dextrose%'
  AND drug NOT LIKE '%Flush%' AND drug NOT LIKE '%Water%'
  AND drug NOT IN ('Bag', 'Iso-Osmotic Dextrose')
GROUP BY drug ORDER BY admissions DESC LIMIT 15
""").to_csv(APP_DATA / "mimic_top_drugs.csv", index=False)

# 3) Drugs in diabetic vs non-diabetic admissions (ICD-9 250.xx; ICD-10 E08-E11, E13)
diab_sql = """
    SELECT DISTINCT hadm_id FROM diagnoses_icd
    WHERE (icd_version = 9 AND icd_code LIKE '250%')
       OR (icd_version = 10 AND (icd_code LIKE 'E08%' OR icd_code LIKE 'E09%'
                              OR icd_code LIKE 'E10%' OR icd_code LIKE 'E11%'
                              OR icd_code LIKE 'E13%'))
"""
n_all = int(q("SELECT COUNT(DISTINCT hadm_id) AS n FROM admissions")["n"][0])
n_dm = int(q(f"SELECT COUNT(*) AS n FROM ({diab_sql})")["n"][0])
dd_ = q(f"""
WITH diabetic AS ({diab_sql})
SELECT p.drug,
       COUNT(DISTINCT p.hadm_id) AS all_adm,
       COUNT(DISTINCT CASE WHEN d.hadm_id IS NOT NULL THEN p.hadm_id END) AS dm_adm
FROM prescriptions p
LEFT JOIN diabetic d ON p.hadm_id = d.hadm_id
WHERE p.drug IS NOT NULL
  AND p.drug NOT LIKE '%Sodium Chloride%' AND p.drug NOT LIKE '%Dextrose%'
  AND p.drug NOT LIKE '%Flush%' AND p.drug <> 'Bag'
GROUP BY p.drug
HAVING all_adm >= 40
""")
dd_["pct_diabetic"] = (dd_["dm_adm"] / n_dm * 100).round(1)
dd_["pct_non_diabetic"] = ((dd_["all_adm"] - dd_["dm_adm"]) / (n_all - n_dm) * 100).round(1)
dd_["gap"] = (dd_["pct_diabetic"] - dd_["pct_non_diabetic"]).round(1)
dd_.sort_values("gap", ascending=False).head(12).to_csv(APP_DATA / "mimic_diabetic_drugs.csv", index=False)

# 4) Length of stay by admission type
q("""
SELECT admission_type, COUNT(*) AS admissions,
       ROUND(AVG(julianday(dischtime) - julianday(admittime)), 1) AS avg_los_days
FROM admissions WHERE dischtime IS NOT NULL
GROUP BY admission_type ORDER BY admissions DESC
""").to_csv(APP_DATA / "mimic_los.csv", index=False)

# 5) Insulin: orders versus recorded administrations
q("""
WITH ordered AS (SELECT DISTINCT hadm_id FROM prescriptions WHERE LOWER(drug) LIKE '%insulin%'),
administered AS (SELECT DISTINCT hadm_id FROM emar
                 WHERE LOWER(medication) LIKE '%insulin%' AND event_txt = 'Administered'),
emar_adm AS (SELECT DISTINCT hadm_id FROM emar)
SELECT
  (SELECT COUNT(*) FROM ordered)                                       AS insulin_ordered,
  (SELECT COUNT(*) FROM administered)                                  AS insulin_administered,
  (SELECT COUNT(*) FROM ordered o JOIN administered a USING (hadm_id)) AS both_,
  (SELECT COUNT(*) FROM ordered o WHERE o.hadm_id IN (SELECT hadm_id FROM emar_adm)) AS ordered_in_emar
""").to_csv(APP_DATA / "mimic_insulin.csv", index=False)

# 6) Headline numbers: size, validation and readmission
adm = q("SELECT COUNT(*) AS n, COUNT(DISTINCT subject_id) AS patients FROM admissions").iloc[0]
rd = q("""
WITH o AS (
  SELECT subject_id, hospital_expire_flag, dischtime,
         LEAD(admittime)      OVER (PARTITION BY subject_id ORDER BY admittime) AS next_admit,
         LEAD(admission_type) OVER (PARTITION BY subject_id ORDER BY admittime) AS next_type
  FROM admissions)
SELECT COUNT(*) AS discharged_alive,
       SUM(julianday(next_admit) - julianday(dischtime) BETWEEN 0 AND 30) AS readmit_any,
       SUM(julianday(next_admit) - julianday(dischtime) BETWEEN 0 AND 30
           AND next_type NOT IN ('ELECTIVE', 'SURGICAL SAME DAY ADMISSION')) AS readmit_unplanned
FROM o WHERE hospital_expire_flag = 0
""").iloc[0]
json.dump({
    "admissions": int(adm["n"]), "patients": int(adm["patients"]), "with_diabetes": n_dm,
    "discharged_alive": int(rd["discharged_alive"]),
    "readmit_any": int(rd["readmit_any"]), "readmit_unplanned": int(rd["readmit_unplanned"]),
    "tables_checked": int(len(check)), "table_mismatches": int((~check["match"]).sum()),
}, open(APP_DATA / "mimic_kpis.json", "w"), indent=2)

print(sorted(p.name for p in APP_DATA.iterdir() if p.name.startswith("mimic")))